# Model Testing - Doctor Recommendation
This notebook runs a sample recommendation flow from input to ranked output.

In [ ]:
from pathlib import Path
import pickle as pk
import pandas as pd

BASE_DIR = Path.cwd().resolve().parent
MODEL_PATH = BASE_DIR / 'models' / 'doctor_ai_full_package.pkl'

with open(MODEL_PATH, 'rb') as f:
    package = pk.load(f)

model = package['model']
le_spec = package['le_spec']
le_hosp = package['le_hosp']
feature_encoders = package['feature_encoders']
features = package['features']

In [ ]:
surgeon_encoded = le_spec.transform(['Surgeon'])[0]
sample_data = {
    'doctor_name': ['Dr. Md. Nahid Hasan', 'Dr. Md. Fazlul Haque Qasem', 'Dr. AMM Mukarrabin', 'Dr. Md. Akter'],
    'rating_avg': [4.53, 3.64, 3.91, 3.52],
    'rating_count': [100, 85, 120, 95],
    'experience_years': [17, 22, 14, 17],
    'consultation_fees': [2000, 600, 1800, 500],
    'hospital_name': ['Lancet Diagnostic Center', 'Lancet Diagnostic Center', 'Community Based Medical College Hospital', 'Gazi Medical College Hospital'],
    'hospital_type': ['Diagnostic Center', 'Diagnostic Center', 'Medical College Hospital', 'Medical College Hospital'],
    'full_address': ['837 Hospital Lane, Bakerganj, Barisal', '424/A Bakerganj Road, Barisal', '767/A Bakerganj Road, Barisal', '777 Hospital Lane, Bakerganj, Barisal'],
    'district': ['Barisal', 'Barisal', 'Barisal', 'Barisal'],
    'thana': ['Bakerganj', 'Bakerganj', 'Bakerganj', 'Bakerganj'],
    'specialization_group': [surgeon_encoded, surgeon_encoded, surgeon_encoded, surgeon_encoded],
    'online_consultation': [0, 0, 0, 0],
    'emergency_service': [0, 0, 0, 0]
}

df = pd.DataFrame(sample_data)
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')

In [ ]:
def recommend_from_pickle(user_input, input_df, model, top_n=5):
    temp_df = input_df.copy()

    temp_df = temp_df[
        (temp_df['district'] == user_input['district']) &
        (temp_df['thana'] == user_input['thana'])
    ]

    temp_df = temp_df[temp_df['consultation_fees'] <= user_input['max_fee']]

    if user_input['online'] == 1:
        temp_df = temp_df[temp_df['online_consultation'] == 1]

    if user_input['emergency'] == 1:
        temp_df = temp_df[temp_df['emergency_service'] == 1]

    try:
        spec_encoded = le_spec.transform([user_input['specialization']])[0]
        temp_df = temp_df[temp_df['specialization_group'] == spec_encoded]
    except Exception:
        return 'Specialization not found'

    if len(temp_df) == 0:
        return 'No doctors found'

    x_temp = temp_df[features].copy()

    if 'hospital_type' in x_temp.columns:
        hosp_type_mapping = {label: idx for idx, label in enumerate(le_hosp.classes_)}
        x_temp['hospital_type'] = x_temp['hospital_type'].astype(str).map(hosp_type_mapping).fillna(-1).astype(int)

    for col in x_temp.columns:
        if col in feature_encoders:
            encoder = feature_encoders[col]
            label_mapping = {label: idx for idx, label in enumerate(encoder.classes_)}
            x_temp[col] = x_temp[col].astype(str).map(label_mapping).fillna(-1).astype(int)

    temp_df['predicted_score'] = model.predict(x_temp)
    result = temp_df.sort_values(by='predicted_score', ascending=False)

    return result[[
        'doctor_name',
        'rating_avg',
        'experience_years',
        'consultation_fees',
        'predicted_score',
        'hospital_name',
        'full_address'
    ]].head(top_n)

In [ ]:
user_input = {
    'district': 'Barisal',
    'thana': 'Bakerganj',
    'specialization': 'Surgeon',
    'max_fee': 2000,
    'online': 0,
    'emergency': 0
}

print('=' * 100)
print('DOCTOR RECOMMENDATION RESULTS')
print('=' * 100)
print(
    f"Search Criteria: {user_input['district']}, {user_input['thana']} | "
    f"Specialization: {user_input['specialization']} | Max Fee: {user_input['max_fee']}"
 )
print('=' * 100)

result = recommend_from_pickle(user_input, df, model, top_n=5)

if isinstance(result, str):
    print(f'Status: {result}')
else:
    print(result.to_string(index=False))
print('=' * 100)